# Pruebas de Endpoints (Regresión)

Este notebook valida endpoints del backend sin cambiar la lógica de chunking ya probada en `preparacion.ipynb`.

Objetivo:
- Probar salud del servicio.
- Probar ingesta.
- Verificar que métricas de chunking no se desvíen del baseline validado.
- Probar consulta RAG y trazabilidad.
- Emitir reporte rápido de regresión.

## 1. Configurar entorno y variables base de testing

Cargamos `.env`, configuramos URL base y validamos prerequisitos mínimos para cortar temprano si falta algo crítico.

In [5]:
from pathlib import Path
import os
import json
import time
from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path, override=False)

BASE_URL = os.getenv("API_BASE_URL", "http://127.0.0.1:8000")
API_PREFIX = "/api/v1"
TIMEOUT = float(os.getenv("TEST_TIMEOUT_SECONDS", "30"))
MODO_LOCAL = BASE_URL.startswith("http://127.0.0.1") or BASE_URL.startswith("http://localhost")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

BASELINE = {
    # Ley canónica de referencia para paridad con preparacion.ipynb
    "ley_numero": 20744,
    "expected_chunks": 276,
    "min_chunks": 200,
    "max_chunks": 400,
    "required_metadata_keys": [
        "law_id",
        "infoleg_id",
        "source_actualizado",
        "sha256_hash",
    ],
}

print(f"BASE_URL: {BASE_URL}")
print(f"MODO_LOCAL: {MODO_LOCAL}")
print(f"TIMEOUT: {TIMEOUT}")
print("OPENAI_API_KEY:", "OK" if OPENAI_API_KEY else "NO CONFIGURADA")

BASE_URL: http://127.0.0.1:8000
MODO_LOCAL: True
TIMEOUT: 30.0
OPENAI_API_KEY: OK


## 2. Importar cliente HTTP y utilidades de aserción

Helpers consistentes para request, logging de respuesta y asserts.

In [14]:
import requests


def _url(path: str) -> str:
    if not path.startswith("/"):
        path = "/" + path
    return f"{BASE_URL}{path}"


def request_json(method: str, path: str, payload: dict | None = None, expected_status: int | None = None):
    t0 = time.perf_counter()
    try:
        response = requests.request(
            method=method.upper(),
            url=_url(path),
            json=payload,
            timeout=TIMEOUT,
        )
    except requests.RequestException as exc:
        latency_ms = round((time.perf_counter() - t0) * 1000, 2)
        data = {"error": str(exc)}
        print(f"[{method.upper()}] {path} -> CONNECTION ERROR ({latency_ms} ms)")
        if expected_status is not None:
            raise AssertionError(f"Status esperado {expected_status}, pero hubo error de conexión: {exc}") from exc
        return {
            "status_code": 0,
            "latency_ms": latency_ms,
            "data": data,
        }

    latency_ms = round((time.perf_counter() - t0) * 1000, 2)

    try:
        data = response.json()
    except Exception:
        data = {"raw": response.text}

    print(f"[{method.upper()}] {path} -> {response.status_code} ({latency_ms} ms)")
    if response.status_code >= 400:
        print("ERROR BODY:", json.dumps(data, ensure_ascii=False, indent=2))

    if expected_status is not None:
        assert response.status_code == expected_status, (
            f"Status esperado {expected_status}, obtenido {response.status_code}. Body: {data}"
        )

    return {
        "status_code": response.status_code,
        "latency_ms": latency_ms,
        "data": data,
    }


def assert_keys(obj: dict, required_keys: list[str], context: str = ""):
    missing = [k for k in required_keys if k not in obj]
    assert not missing, f"Faltan keys en {context}: {missing}"


def ensure_runtime_state() -> dict[str, object]:
    if "BASELINE" not in globals():
        raise RuntimeError("Primero ejecuta la celda de configuración base (paso 1).")

    globals()["LAW_ID"] = BASELINE["ley_numero"]

    globals()["payload_ingest_law"] = globals().get("payload_ingest_law") or {
        "ley_numero": globals()["LAW_ID"],
        "replace_existing": True,
    }

    globals()["payload_answer"] = globals().get("payload_answer") or {
        "question": "Que establece la ley 27275 sobre acceso a la información pública?",
        "top_k": 3,
    }

    globals()["payload_truth"] = globals().get("payload_truth") or {
        "statement": "La ley 27275 regula el acceso a la información pública en Argentina.",
        "top_k": 3,
    }

    globals()["payload_agent_chat"] = globals().get("payload_agent_chat") or {
        "prompt": "Es cierto que la Ley 27.275 regula el acceso a la información pública en Argentina?",
    }

    globals()["run_artifacts"] = globals().get("run_artifacts") or {
        "health": None,
        "ingest": None,
        "answer": None,
        "truth_index": None,
        "agent_chat": None,
        "chunking_validation": None,
    }

    return {
        "LAW_ID": globals()["LAW_ID"],
        "payload_ingest_law": globals()["payload_ingest_law"],
        "payload_answer": globals()["payload_answer"],
        "payload_truth": globals()["payload_truth"],
        "payload_agent_chat": globals()["payload_agent_chat"],
        "run_artifacts": globals()["run_artifacts"],
    }

## 3. Definir fixtures reutilizables (payloads, paths, headers)

Centralizamos payloads para evitar duplicación entre celdas.

In [15]:
runtime_state = ensure_runtime_state()

LAW_ID = runtime_state["LAW_ID"]
payload_ingest_law = runtime_state["payload_ingest_law"]
payload_answer = runtime_state["payload_answer"]
payload_truth = runtime_state["payload_truth"]
payload_agent_chat = runtime_state["payload_agent_chat"]
run_artifacts = runtime_state["run_artifacts"]

print("Fixtures listos.")
print(json.dumps({
    "payload_ingest_law": payload_ingest_law,
    "payload_answer": payload_answer,
    "payload_truth": payload_truth,
    "payload_agent_chat": payload_agent_chat,
}, ensure_ascii=False, indent=2))

Fixtures listos.
{
  "payload_ingest_law": {
    "ley_numero": 20744,
    "replace_existing": true
  },
  "payload_answer": {
    "question": "Que establece la ley 27275 sobre acceso a la información pública?",
    "top_k": 3
  },
  "payload_truth": {
    "statement": "La ley 27275 regula el acceso a la información pública en Argentina.",
    "top_k": 3
  },
  "payload_agent_chat": {
    "prompt": "Es cierto que la Ley 27.275 regula el acceso a la información pública en Argentina?"
  }
}


## 4. Probar endpoint de health/status

In [9]:
run_artifacts = ensure_runtime_state()["run_artifacts"]

health = request_json("GET", f"{API_PREFIX}/health")
run_artifacts["health"] = health

if health["status_code"] == 200:
    assert health["latency_ms"] < 4000, f"Health demasiado lento: {health['latency_ms']} ms"
    assert isinstance(health["data"], dict), "Health debe responder JSON"
    print("Health OK")
    print(json.dumps(health["data"], ensure_ascii=False, indent=2))
else:
    print("Health no disponible. Verifica que el backend esté levantado en BASE_URL antes de continuar.")
    print(json.dumps(health["data"], ensure_ascii=False, indent=2))

[GET] /api/v1/health -> 200 (3.91 ms)
Health OK
{
  "status": "ok"
}


## 5. Probar endpoint de ingesta con documento de prueba

In [10]:
runtime_state = ensure_runtime_state()
payload_ingest_law = runtime_state["payload_ingest_law"]
run_artifacts = runtime_state["run_artifacts"]

if run_artifacts.get("health", {}).get("status_code") != 200:
    print("Saltando ingesta: health no está OK.")
    run_artifacts["ingest"] = {
        "status_code": 0,
        "latency_ms": 0,
        "data": {"error": "Health no disponible"},
    }
else:
    ingest = request_json(
        "POST",
        f"{API_PREFIX}/rag/ingest/law",
        payload=payload_ingest_law,
        expected_status=200,
    )
    run_artifacts["ingest"] = ingest

    required_ingest_keys = [
        "document_id",
        "source_uri",
        "law_id",
        "infoleg_id",
        "chunks_inserted",
        "source_norma",
        "source_actualizado",
        "sha256_hash",
    ]
    assert_keys(ingest["data"], required_ingest_keys, "ingest/law")
    assert ingest["data"]["law_id"] == runtime_state["LAW_ID"], "law_id devuelto no coincide con el payload"
    assert isinstance(ingest["data"]["chunks_inserted"], int), "chunks_inserted debe ser int"
    assert ingest["data"]["chunks_inserted"] >= 1, "La ingesta no creó chunks"

    print("Ingesta OK")
    print(json.dumps(ingest["data"], ensure_ascii=False, indent=2))

[POST] /api/v1/rag/ingest/law -> 200 (3191.31 ms)
Ingesta OK
{
  "document_id": "infoleg-25552-1e051cf62df4",
  "source_uri": "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm",
  "law_id": 20744,
  "infoleg_id": 25552,
  "chunks_inserted": 276,
  "source_norma": "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/norma.htm",
  "source_actualizado": "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm",
  "sha256_hash": "1e051cf62df49b25c5f5796295e8bdccd738e65112939ee30425e3a2c33b89a8",
  "db_path": "app/data/rag.sqlite3"
}


## 6. Validar que el chunking del endpoint mantenga métricas esperadas

No redefinimos chunking: comparamos contra baseline/tolerancias para detectar regresiones del comportamiento ya validado.

In [11]:
ingest_item = run_artifacts.get("ingest")

if not ingest_item or ingest_item.get("status_code") != 200:
    print("Saltando validación de chunking: no hay ingesta exitosa.")
    run_artifacts["chunking_validation"] = {
        "skipped": True,
        "reason": "Ingesta no disponible",
    }
else:
    chunk_count = int(ingest_item["data"]["chunks_inserted"])

    validation = {
        "law_id": ingest_item["data"].get("law_id"),
        "chunk_count": chunk_count,
        "expected_chunks": BASELINE["expected_chunks"],
        "baseline_min": BASELINE["min_chunks"],
        "baseline_max": BASELINE["max_chunks"],
        "source_actualizado": ingest_item["data"].get("source_actualizado"),
        "has_sha256": bool(ingest_item["data"].get("sha256_hash")),
    }

    assert chunk_count >= BASELINE["min_chunks"], (
        f"Regresión: chunk_count {chunk_count} < min {BASELINE['min_chunks']}"
    )
    assert chunk_count <= BASELINE["max_chunks"], (
        f"Regresión: chunk_count {chunk_count} > max {BASELINE['max_chunks']}"
    )
    assert chunk_count == BASELINE["expected_chunks"], (
        f"Regresión respecto a preparación: esperado {BASELINE['expected_chunks']}, obtenido {chunk_count}"
    )

    for key in BASELINE["required_metadata_keys"]:
        assert ingest_item["data"].get(key) not in (None, ""), f"Falta metadata obligatoria: {key}"

    run_artifacts["chunking_validation"] = validation
    print("Validación chunking OK")
    print(json.dumps(validation, ensure_ascii=False, indent=2))

Validación chunking OK
{
  "law_id": 20744,
  "chunk_count": 276,
  "expected_chunks": 276,
  "baseline_min": 200,
  "baseline_max": 400,
  "source_actualizado": "https://servicios.infoleg.gob.ar/infolegInternet/anexos/25000-29999/25552/texact.htm",
  "has_sha256": true
}


## 7. Probar endpoint de consulta RAG y verificar trazabilidad

In [12]:
if run_artifacts.get("health", {}).get("status_code") != 200:
    print("Saltando endpoints RAG de consulta: health no está OK.")
    run_artifacts["answer"] = {"status_code": 0, "latency_ms": 0, "data": {"error": "Health no disponible"}}
    run_artifacts["truth_index"] = {"status_code": 0, "latency_ms": 0, "data": {"error": "Health no disponible"}}
else:
    answer = request_json(
        "POST",
        f"{API_PREFIX}/rag/answer",
        payload=payload_answer,
    )
    run_artifacts["answer"] = answer

    if answer["status_code"] == 200:
        assert_keys(answer["data"], ["answer", "sources", "question"], "rag/answer")
        assert isinstance(answer["data"]["sources"], list), "sources debe ser lista"
        print("rag/answer OK")
    else:
        print("rag/answer no disponible o falló en este entorno. Se conserva evidencia para reporte.")

    truth_idx = request_json(
        "POST",
        f"{API_PREFIX}/rag/truth-index",
        payload=payload_truth,
    )
    run_artifacts["truth_index"] = truth_idx

    if truth_idx["status_code"] == 200:
        assert_keys(truth_idx["data"], ["indice_verdad", "justificacion", "statement"], "rag/truth-index")
        print("rag/truth-index OK")
    else:
        print("rag/truth-index no disponible o falló en este entorno. Se conserva evidencia para reporte.")

[POST] /api/v1/rag/answer -> 200 (14206.63 ms)
rag/answer OK
[POST] /api/v1/rag/truth-index -> 500 (3.38 ms)
ERROR BODY: {
  "detail": "'LocalRAGService' object has no attribute 'calculate_truth_index'"
}
rag/truth-index no disponible o falló en este entorno. Se conserva evidencia para reporte.


In [ ]:
runtime_state = ensure_runtime_state()
payload_agent_chat = runtime_state["payload_agent_chat"]
run_artifacts = runtime_state["run_artifacts"]

if run_artifacts.get("health", {}).get("status_code") != 200:
    print("Saltando agent/chat: health no está OK.")
    run_artifacts["agent_chat"] = {
        "status_code": 0,
        "latency_ms": 0,
        "data": {"error": "Health no disponible"},
    }
else:
    agent_chat = request_json(
        "POST",
        f"{API_PREFIX}/agent/chat",
        payload=payload_agent_chat,
        expected_status=200,
    )
    run_artifacts["agent_chat"] = agent_chat

    required_agent_keys = [
        "id",
        "query",
        "verdict",
        "summary_ia",
        "hash",
        "source_law",
        "source_url",
        "original_text",
        "highlights",
        "news_context",
        "chunks_used",
        "used_model",
    ]
    assert_keys(agent_chat["data"], required_agent_keys, "agent/chat")
    assert agent_chat["data"]["query"] == payload_agent_chat["prompt"], "La query devuelta no coincide con el prompt enviado"
    assert isinstance(agent_chat["data"]["summary_ia"], str) and agent_chat["data"]["summary_ia"].strip(), "summary_ia debe tener contenido"
    assert isinstance(agent_chat["data"]["highlights"], list), "highlights debe ser lista"
    assert isinstance(agent_chat["data"]["news_context"], list), "news_context debe ser lista"
    assert isinstance(agent_chat["data"]["chunks_used"], list), "chunks_used debe ser lista"
    if agent_chat["data"]["chunks_used"]:
        first_chunk = agent_chat["data"]["chunks_used"][0]
        assert isinstance(first_chunk, dict), "cada chunk usado debe ser un objeto"
        assert "text" in first_chunk and "source" in first_chunk and "metadata" in first_chunk, "chunk_used incompleto para UI"
    assert isinstance(agent_chat["data"]["used_model"], str) and agent_chat["data"]["used_model"].strip(), "used_model debe informar origen/modelo"

    print("agent/chat OK")
    print(json.dumps(agent_chat["data"], ensure_ascii=False, indent=2))

[POST] /api/v1/agent/chat -> 200 (6922.27 ms)
agent/chat OK
{
  "id": "AUD-2026-E39559",
  "query": "Es cierto que la Ley 27.275 regula el acceso a la información pública en Argentina?",
  "verdict": "VERDADERO",
  "summary_ia": "La Ley 27.275 efectivamente regula el acceso a la información pública en Argentina, estableciendo derechos y procedimientos para su ejercicio. Esta ley busca garantizar la transparencia y el derecho de los ciudadanos a acceder a información pública.",
  "hash": "sha256:bcde971a22eca5927ee5f8d35ea3d0a9394d8bf500816885dade8f8d292a7417",
  "source_law": "Ley 27.275 - TÍTULO I",
  "source_url": "https://servicios.infoleg.gob.ar/infolegInternet/anexos/265000-269999/265949/texact.htm",
  "original_text": "DERECHO DE ACCESO A LA INFORMACIÓN PÚBLICA Ley 27275 Objeto. Excepciones. Alcances.",
  "highlights": [
    "Ley 27.275",
    "acceso a la información pública"
  ],
  "news_context": [],
  "used_model": "gpt-4o-mini"
}


## 8. Probar endpoint del agente con la consulta del usuario

Validamos el endpoint real que consume el frontend y verificamos el contrato de respuesta del agente.

## 9. Ejecutar suite rápida y reporte de resultados

Consolidamos pass/fail, endpoint, detalle y latencia; además exportamos un resumen JSON para seguimiento de regresiones, incluyendo la respuesta del agente al prompt del usuario.

In [17]:
rows = []

for name, path in [
    ("health", f"{API_PREFIX}/health"),
    ("ingest_law", f"{API_PREFIX}/rag/ingest/law"),
    ("answer", f"{API_PREFIX}/rag/answer"),
    ("truth_index", f"{API_PREFIX}/rag/truth-index"),
    ("agent_chat", f"{API_PREFIX}/agent/chat"),
]:
    item = run_artifacts.get(name if name != "ingest_law" else "ingest")
    if item is None:
        rows.append({
            "test": name,
            "endpoint": path,
            "pass": False,
            "status_code": None,
            "latency_ms": None,
            "detail": "No ejecutado",
        })
        continue

    status_code = item.get("status_code")
    ok = status_code is not None and int(status_code) >= 200 and int(status_code) < 400
    rows.append({
        "test": name,
        "endpoint": path,
        "pass": ok,
        "status_code": status_code,
        "latency_ms": item.get("latency_ms"),
        "detail": "OK" if ok else str(item.get("data"))[:240],
    })

summary = {
    "total": len(rows),
    "passed": sum(1 for r in rows if r["pass"]),
    "failed": sum(1 for r in rows if not r["pass"]),
    "rows": rows,
    "chunking_validation": run_artifacts.get("chunking_validation"),
    "agent_chat_response": (run_artifacts.get("agent_chat") or {}).get("data"),
}

for r in rows:
    print(f"- {r['test']}: {'PASS' if r['pass'] else 'FAIL'} | {r['status_code']} | {r['latency_ms']} ms")

report_path = Path("app/data/endpoint_test_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("\nResumen:")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"\nReporte exportado en: {report_path}")

- health: PASS | 200 | 3.91 ms
- ingest_law: PASS | 200 | 3191.31 ms
- answer: PASS | 200 | 14206.63 ms
- truth_index: FAIL | 500 | 3.38 ms
- agent_chat: PASS | 200 | 6922.27 ms

Resumen:
{
  "total": 5,
  "passed": 4,
  "failed": 1,
  "rows": [
    {
      "test": "health",
      "endpoint": "/api/v1/health",
      "pass": true,
      "status_code": 200,
      "latency_ms": 3.91,
      "detail": "OK"
    },
    {
      "test": "ingest_law",
      "endpoint": "/api/v1/rag/ingest/law",
      "pass": true,
      "status_code": 200,
      "latency_ms": 3191.31,
      "detail": "OK"
    },
    {
      "test": "answer",
      "endpoint": "/api/v1/rag/answer",
      "pass": true,
      "status_code": 200,
      "latency_ms": 14206.63,
      "detail": "OK"
    },
    {
      "test": "truth_index",
      "endpoint": "/api/v1/rag/truth-index",
      "pass": false,
      "status_code": 500,
      "latency_ms": 3.38,
      "detail": "{'detail': \"'LocalRAGService' object has no attribute 'calcula

## Respuestas esperadas y troubleshooting

Esperado:
- `GET /api/v1/health` responde 200.
- `POST /api/v1/rag/ingest/law` responde 200 y `chunks_inserted >= 1`.
- `POST /api/v1/agent/chat` responde 200 y devuelve `query`, `verdict`, `summary_ia`, `hash` y demás campos del contrato.
- `POST /api/v1/rag/answer` y `POST /api/v1/rag/truth-index` pueden fallar si falta configuración de modelo/credenciales.

Si falla `agent/chat`:
1. Verificar que el backend esté levantado en `BASE_URL`.
2. Revisar que exista contexto RAG cargado si esperás una respuesta fundamentada en normas.
3. Revisar `OPENAI_API_KEY` y el modelo configurado si la respuesta falla o vuelve vacía.
4. Si responde `mock`, confirmar si ese entorno está pensado para pruebas sin LLM real.

Si falla ingesta:
1. Verificar que el backend esté levantado en `BASE_URL`.
2. Revisar conectividad saliente a InfoLEG desde el host del backend.
3. Revisar logs del servidor para errores de parseo o timeouts.
4. Comparar métricas de chunking con baseline antes de tocar la lógica de chunking.

Nota importante:
- Este notebook está pensado para proteger el comportamiento actual. Si el chunking del notebook ya funciona, el backend debe alinearse con ese baseline en lugar de redefinir reglas nuevas.

In [ ]:
print('smoke test notebook ok')